# Day 06 — Pandas: Series, DataFrames, Filtering, Sorting & Missing Values


In [1]:
import numpy as np
import pandas as pd

print("pandas version:", pd.__version__)

# Cleaner display for this notebook
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

pandas version: 3.0.5


---
## 1. Series

A **Series** is a one-dimensional labeled array: values plus an **index**.

### 1.1 Creating a Series

In [2]:
s = pd.Series([10, 20, 30, 40])
print(s)

# With a custom index
prices = pd.Series([1.2, 0.8, 2.5], index=["apple", "banana", "cherry"], name="price")
print(prices)

# From a dictionary (keys become the index)
population = pd.Series({"Lahore": 13, "Karachi": 20, "Islamabad": 1.2})
print(population)

# From a scalar
print(pd.Series(0, index=["a", "b", "c"]))

0    10
1    20
2    30
3    40
dtype: int64
apple     1.2
banana    0.8
cherry    2.5
Name: price, dtype: float64
Lahore       13.0
Karachi      20.0
Islamabad     1.2
dtype: float64
a    0
b    0
c    0
dtype: int64


### 1.2 Attributes

In [3]:
print("values:", prices.values)
print("index :", prices.index.tolist())
print("dtype :", prices.dtype)
print("name  :", prices.name)
print("shape :", prices.shape)
print("size  :", prices.size)

values: [1.2 0.8 2.5]
index : ['apple', 'banana', 'cherry']
dtype : float64
name  : price
shape : (3,)
size  : 3


### 1.3 Selecting from a Series

In [4]:
print(prices["banana"])              # by label
print(prices.iloc[0])                # by position
print(prices[["apple", "cherry"]])   # several labels
print(prices["apple":"banana"])      # label slice (end INCLUDED)
print(prices.iloc[0:2])              # position slice (end excluded)

0.8
1.2
apple     1.2
cherry    2.5
Name: price, dtype: float64
apple     1.2
banana    0.8
Name: price, dtype: float64
apple     1.2
banana    0.8
Name: price, dtype: float64


### 1.4 Vectorized operations and alignment

In [5]:
print(prices * 100)
print(prices.round(0))
print(prices[prices > 1])

# Operations align on index labels, not position
a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20, 30], index=["y", "z", "w"])
print(a + b)       # labels without a match become NaN

apple     120.0
banana     80.0
cherry    250.0
Name: price, dtype: float64
apple     1.0
banana    1.0
cherry    2.0
Name: price, dtype: float64
apple     1.2
cherry    2.5
Name: price, dtype: float64
w     NaN
x     NaN
y    12.0
z    23.0
dtype: float64


### 1.5 Useful Series methods

In [6]:
colors = pd.Series(["red", "blue", "red", "green", "blue", "red"])

print(colors.value_counts())
print(colors.unique())
print(colors.nunique())
print(colors.str.upper().tolist())

nums = pd.Series([4, 8, 15, 16, 23, 42])
print(nums.describe())
print(nums.mean(), nums.median(), nums.std().round(2))
print(nums.apply(lambda x: x ** 2).tolist())
print(nums.map({4: "four", 8: "eight"}).tolist())

red      3
blue     2
green    1
Name: count, dtype: int64
<StringArray>
['red', 'blue', 'green']
Length: 3, dtype: str
3
['RED', 'BLUE', 'RED', 'GREEN', 'BLUE', 'RED']
count     6.000000
mean     18.000000
std      13.490738
min       4.000000
25%       9.750000
50%      15.500000
75%      21.250000
max      42.000000
dtype: float64
18.0 15.5 13.49
[16, 64, 225, 256, 529, 1764]
['four', 'eight', nan, nan, nan, nan]


---
## 2. DataFrames

A **DataFrame** is a 2-D table: rows and columns, each column being a Series. Think of a spreadsheet or SQL table.

### 2.1 Creating a DataFrame

In [7]:
# From a dictionary of columns
df = pd.DataFrame({
    "name": ["Ana", "Ben", "Cy", "Dia", "Eli"],
    "age": [23, 31, 28, 35, 26],
    "city": ["Lahore", "Karachi", "Lahore", "Islamabad", "Karachi"],
    "salary": [52000, 61000, 58000, 75000, 49000],
})
print(df)

# From a list of dictionaries (one per row)
rows = [{"id": 1, "item": "pen"}, {"id": 2, "item": "book"}]
print(pd.DataFrame(rows))

# From a NumPy array
print(pd.DataFrame(np.arange(6).reshape(2, 3), columns=["a", "b", "c"]))

  name  age       city  salary
0  Ana   23     Lahore   52000
1  Ben   31    Karachi   61000
2   Cy   28     Lahore   58000
3  Dia   35  Islamabad   75000
4  Eli   26    Karachi   49000
   id  item
0   1   pen
1   2  book
   a  b  c
0  0  1  2
1  3  4  5


### 2.2 Inspecting a DataFrame

In [8]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.head(3))
print(df.tail(2))
df.info()

(5, 4)
['name', 'age', 'city', 'salary']
name        str
age       int64
city        str
salary    int64
dtype: object
  name  age     city  salary
0  Ana   23   Lahore   52000
1  Ben   31  Karachi   61000
2   Cy   28   Lahore   58000
  name  age       city  salary
3  Dia   35  Islamabad   75000
4  Eli   26    Karachi   49000
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    5 non-null      str  
 1   age     5 non-null      int64
 2   city    5 non-null      str  
 3   salary  5 non-null      int64
dtypes: int64(2), str(2)
memory usage: 292.0 bytes


In [9]:
print(df.describe())                        # numeric summary
print(df.describe(include="object"))        # text columns

             age        salary
count   5.000000      5.000000
mean   28.600000  59000.000000
std     4.615192  10124.228366
min    23.000000  49000.000000
25%    26.000000  52000.000000
50%    28.000000  58000.000000
75%    31.000000  61000.000000
max    35.000000  75000.000000
       name    city
count     5       5
unique    5       3
top     Ana  Lahore
freq      1       2


/tmp/ipykernel_205/2603430192.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.describe(include="object"))        # text columns


### 2.3 Selecting columns

In [10]:
print(df["name"])                 # one column -> Series
print(df[["name", "salary"]])     # several columns -> DataFrame
print(type(df["name"]), type(df[["name"]]))

0    Ana
1    Ben
2     Cy
3    Dia
4    Eli
Name: name, dtype: str
  name  salary
0  Ana   52000
1  Ben   61000
2   Cy   58000
3  Dia   75000
4  Eli   49000
<class 'pandas.Series'> <class 'pandas.DataFrame'>


### 2.4 Selecting rows: `loc` vs `iloc`

In [11]:
# iloc = by integer POSITION (end excluded)
print(df.iloc[0])                 # first row
print(df.iloc[1:3])               # rows 1 and 2
print(df.iloc[0, 1])              # row 0, column 1
print(df.iloc[:2, [0, 3]])        # first two rows, columns 0 and 3

name         Ana
age           23
city      Lahore
salary     52000
Name: 0, dtype: object
  name  age     city  salary
1  Ben   31  Karachi   61000
2   Cy   28   Lahore   58000
23
  name  salary
0  Ana   52000
1  Ben   61000


In [12]:
# loc = by LABEL (end included)
indexed = df.set_index("name")
print(indexed)

print(indexed.loc["Ben"])
print(indexed.loc["Ben", "salary"])
print(indexed.loc["Ana":"Cy", ["age", "city"]])

      age       city  salary
name                        
Ana    23     Lahore   52000
Ben    31    Karachi   61000
Cy     28     Lahore   58000
Dia    35  Islamabad   75000
Eli    26    Karachi   49000
age            31
city      Karachi
salary      61000
Name: Ben, dtype: object
61000
      age     city
name              
Ana    23   Lahore
Ben    31  Karachi
Cy     28   Lahore


### 2.5 Adding, changing and removing columns

In [13]:
d = df.copy()

d["bonus"] = d["salary"] * 0.10                       # new column from math
d["senior"] = d["age"] >= 30                          # boolean column
d["name_upper"] = d["name"].str.upper()
d.loc[d["name"] == "Ana", "salary"] = 55000           # change a value

d = d.rename(columns={"name_upper": "NAME"})
d = d.drop(columns=["NAME"])                          # remove a column
print(d)

  name  age       city  salary   bonus  senior
0  Ana   23     Lahore   55000  5200.0   False
1  Ben   31    Karachi   61000  6100.0    True
2   Cy   28     Lahore   58000  5800.0   False
3  Dia   35  Islamabad   75000  7500.0    True
4  Eli   26    Karachi   49000  4900.0   False


### 2.6 Adding and removing rows

In [14]:
d = df.copy()

new_row = pd.DataFrame([{"name": "Fay", "age": 29, "city": "Lahore", "salary": 63000}])
d = pd.concat([d, new_row], ignore_index=True)        # append rows
print(d.tail(2))

d = d.drop(index=[0, 1])                              # drop rows by label
d = d.reset_index(drop=True)                          # tidy the index again
print(d)

  name  age     city  salary
4  Eli   26  Karachi   49000
5  Fay   29   Lahore   63000
  name  age       city  salary
0   Cy   28     Lahore   58000
1  Dia   35  Islamabad   75000
2  Eli   26    Karachi   49000
3  Fay   29     Lahore   63000


### 2.7 Reading and writing files

In [15]:
import tempfile, os

tmp = tempfile.mkdtemp()
csv_path = os.path.join(tmp, "people.csv")

df.to_csv(csv_path, index=False)
back = pd.read_csv(csv_path)
print(back.equals(df))
print(back.head(2))

# Others: read_excel / to_excel, read_json / to_json, read_sql, read_parquet

True
  name  age     city  salary
0  Ana   23   Lahore   52000
1  Ben   31  Karachi   61000


---
## 3. Filtering

We'll use a slightly larger dataset for the next sections.

In [17]:
employees = pd.DataFrame({
    "name":   ["Ana", "Ben", "Cy", "Dia", "Eli", "Fay", "Gus", "Hira", "Ivan", "Jia"],
    "dept":   ["Sales", "IT", "IT", "HR", "Sales", "IT", "HR", "Sales", "IT", "HR"],
    "age":    [23, 31, 28, 35, 26, 41, 30, 29, 38, 33],
    "salary": [52000, 61000, 58000, 75000, 49000, 82000, 54000, 56000, 77000, 64000],
    "years":  [1, 6, 4, 10, 2, 15, 5, 3, 12, 8],
})
employees

,name,dept,age,salary,years
0,Ana,Sales,23,52000,1
1,Ben,IT,31,61000,6
2,Cy,IT,28,58000,4
3,Dia,HR,35,75000,10
4,Eli,Sales,26,49000,2
5,Fay,IT,41,82000,15
6,Gus,HR,30,54000,5
7,Hira,Sales,29,56000,3
8,Ivan,IT,38,77000,12
9,Jia,HR,33,64000,8


### 3.1 Boolean conditions

In [18]:
print(employees[employees["salary"] > 60000])
print(employees[employees["dept"] == "IT"])
print(employees[employees["age"] < 30]["name"].tolist())

   name dept  age  salary  years
1   Ben   IT   31   61000      6
3   Dia   HR   35   75000     10
5   Fay   IT   41   82000     15
8  Ivan   IT   38   77000     12
9   Jia   HR   33   64000      8
   name dept  age  salary  years
1   Ben   IT   31   61000      6
2    Cy   IT   28   58000      4
5   Fay   IT   41   82000     15
8  Ivan   IT   38   77000     12
['Ana', 'Cy', 'Eli', 'Hira']


### 3.2 Combining conditions with `&`, `|`, `~`

In [19]:
# Parentheses are required around each condition
print(employees[(employees["dept"] == "IT") & (employees["salary"] > 60000)])

print(employees[(employees["dept"] == "HR") | (employees["age"] < 25)])

print(employees[~(employees["dept"] == "IT")])        # NOT IT

   name dept  age  salary  years
1   Ben   IT   31   61000      6
5   Fay   IT   41   82000     15
8  Ivan   IT   38   77000     12
  name   dept  age  salary  years
0  Ana  Sales   23   52000      1
3  Dia     HR   35   75000     10
6  Gus     HR   30   54000      5
9  Jia     HR   33   64000      8
   name   dept  age  salary  years
0   Ana  Sales   23   52000      1
3   Dia     HR   35   75000     10
4   Eli  Sales   26   49000      2
6   Gus     HR   30   54000      5
7  Hira  Sales   29   56000      3
9   Jia     HR   33   64000      8


### 3.3 `isin`, `between`, and string matching

In [20]:
print(employees[employees["dept"].isin(["HR", "Sales"])])

print(employees[employees["age"].between(28, 33)])         # inclusive on both ends

print(employees[employees["name"].str.startswith("A")])
print(employees[employees["name"].str.contains("i", case=False)])

   name   dept  age  salary  years
0   Ana  Sales   23   52000      1
3   Dia     HR   35   75000     10
4   Eli  Sales   26   49000      2
6   Gus     HR   30   54000      5
7  Hira  Sales   29   56000      3
9   Jia     HR   33   64000      8
   name   dept  age  salary  years
1   Ben     IT   31   61000      6
2    Cy     IT   28   58000      4
6   Gus     HR   30   54000      5
7  Hira  Sales   29   56000      3
9   Jia     HR   33   64000      8
  name   dept  age  salary  years
0  Ana  Sales   23   52000      1
   name   dept  age  salary  years
3   Dia     HR   35   75000     10
4   Eli  Sales   26   49000      2
7  Hira  Sales   29   56000      3
8  Ivan     IT   38   77000     12
9   Jia     HR   33   64000      8


### 3.4 `query()` — readable filtering

In [21]:
print(employees.query("dept == 'IT' and salary > 60000"))

min_years = 5
print(employees.query("years >= @min_years")[["name", "years"]])

   name dept  age  salary  years
1   Ben   IT   31   61000      6
5   Fay   IT   41   82000     15
8  Ivan   IT   38   77000     12
   name  years
1   Ben      6
3   Dia     10
5   Fay     15
6   Gus      5
8  Ivan     12
9   Jia      8


### 3.5 Filtering and selecting columns together

In [22]:
result = employees.loc[employees["salary"] > 60000, ["name", "dept", "salary"]]
print(result)

# Count and check
mask = employees["years"] >= 10
print("Veterans:", mask.sum(), "| any?", mask.any(), "| all?", mask.all())

   name dept  salary
1   Ben   IT   61000
3   Dia   HR   75000
5   Fay   IT   82000
8  Ivan   IT   77000
9   Jia   HR   64000
Veterans: 3 | any? True | all? False


### 3.6 Unique values and duplicates

In [23]:
print(employees["dept"].unique())
print(employees["dept"].value_counts())

dups = pd.DataFrame({"a": [1, 1, 2, 2, 3], "b": ["x", "x", "y", "z", "w"]})
print(dups.duplicated().tolist())
print(dups.drop_duplicates())
print(dups.drop_duplicates(subset="a", keep="last"))

<StringArray>
['Sales', 'IT', 'HR']
Length: 3, dtype: str
dept
IT       4
Sales    3
HR       3
Name: count, dtype: int64
[False, True, False, False, False]
   a  b
0  1  x
2  2  y
3  2  z
4  3  w
   a  b
1  1  x
3  2  z
4  3  w


---
## 4. Sorting

### 4.1 Sorting by values

In [25]:
print(employees.sort_values("salary"))                        # ascending
print(employees.sort_values("salary", ascending=False).head(3))   # top 3 earners

   name   dept  age  salary  years
4   Eli  Sales   26   49000      2
0   Ana  Sales   23   52000      1
6   Gus     HR   30   54000      5
7  Hira  Sales   29   56000      3
2    Cy     IT   28   58000      4
1   Ben     IT   31   61000      6
9   Jia     HR   33   64000      8
3   Dia     HR   35   75000     10
8  Ivan     IT   38   77000     12
5   Fay     IT   41   82000     15
   name dept  age  salary  years
5   Fay   IT   41   82000     15
8  Ivan   IT   38   77000     12
3   Dia   HR   35   75000     10


### 4.2 Sorting by multiple columns

In [26]:
# Department A->Z, then salary high->low inside each department
out = employees.sort_values(["dept", "salary"], ascending=[True, False])
print(out[["name", "dept", "salary"]])

   name   dept  salary
3   Dia     HR   75000
9   Jia     HR   64000
6   Gus     HR   54000
5   Fay     IT   82000
8  Ivan     IT   77000
1   Ben     IT   61000
2    Cy     IT   58000
7  Hira  Sales   56000
0   Ana  Sales   52000
4   Eli  Sales   49000


### 4.3 Sorting by index, ranking, `nlargest` / `nsmallest`

In [27]:
shuffled = employees.sample(frac=1, random_state=1)
print(shuffled.index.tolist())
print(shuffled.sort_index().index.tolist())

print(employees.nlargest(3, "salary")[["name", "salary"]])
print(employees.nsmallest(2, "age")[["name", "age"]])

employees_ranked = employees.assign(salary_rank=employees["salary"].rank(ascending=False).astype(int))
print(employees_ranked[["name", "salary", "salary_rank"]].sort_values("salary_rank").head(4))

[2, 9, 6, 4, 0, 3, 1, 7, 8, 5]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
   name  salary
5   Fay   82000
8  Ivan   77000
3   Dia   75000
  name  age
0  Ana   23
4  Eli   26
   name  salary  salary_rank
5   Fay   82000            1
8  Ivan   77000            2
3   Dia   75000            3
9   Jia   64000            4


### 4.4 Grouping and summarizing (bonus)

In [28]:
print(employees.groupby("dept")["salary"].mean().round(0))

summary = employees.groupby("dept").agg(
    headcount=("name", "count"),
    avg_salary=("salary", "mean"),
    max_years=("years", "max"),
).sort_values("avg_salary", ascending=False)
print(summary)

dept
HR       64333.0
IT       69500.0
Sales    52333.0
Name: salary, dtype: float64
       headcount    avg_salary  max_years
dept                                     
IT             4  69500.000000         15
HR             3  64333.333333         10
Sales          3  52333.333333          3


---
## 5. Missing Values

Pandas represents missing data as `NaN` (floats), `NaT` (datetimes) or `pd.NA`. Handling them is a core part of data cleaning.

In [30]:
messy = pd.DataFrame({
    "name":   ["Ana", "Ben", "Cy", "Dia", "Eli", "Fay"],
    "age":    [23, np.nan, 28, 35, np.nan, 41],
    "city":   ["Lahore", "Karachi", None, "Islamabad", "Karachi", None],
    "salary": [52000, 61000, np.nan, 75000, 49000, 82000],
    "score":  [88, 92, 75, np.nan, np.nan, 90],
})
messy

,name,age,city,salary,score
0,Ana,23.0,Lahore,52000.0,88.0
1,Ben,NaN,Karachi,61000.0,92.0
2,Cy,28.0,NaN,NaN,75.0
3,Dia,35.0,Islamabad,75000.0,NaN
4,Eli,NaN,Karachi,49000.0,NaN
5,Fay,41.0,NaN,82000.0,90.0


### 5.1 Detecting missing values

In [31]:
print(messy.isna())                       # True where missing
print(messy.isna().sum())                 # count per column
print("total missing:", messy.isna().sum().sum())
print((messy.isna().mean() * 100).round(1))   # % missing per column

print(messy[messy["age"].isna()])             # rows where age is missing
print(messy.notna().sum())

    name    age   city  salary  score
0  False  False  False   False  False
1  False   True  False   False  False
2  False  False   True    True  False
3  False  False  False   False   True
4  False   True  False   False   True
5  False  False   True   False  False
name      0
age       2
city      2
salary    1
score     2
dtype: int64
total missing: 7
name       0.0
age       33.3
city      33.3
salary    16.7
score     33.3
dtype: float64
  name  age     city   salary  score
1  Ben  NaN  Karachi  61000.0   92.0
4  Eli  NaN  Karachi  49000.0    NaN
name      6
age       4
city      4
salary    5
score     4
dtype: int64


### 5.2 Dropping missing values

In [32]:
print("original shape:", messy.shape)

print(messy.dropna().shape)                        # drop rows with ANY missing value
print(messy.dropna(how="all").shape)               # only rows where ALL are missing
print(messy.dropna(subset=["age", "salary"]).shape)  # only look at certain columns
print(messy.dropna(thresh=4).shape)                # keep rows with at least 4 non-null
print(messy.dropna(axis=1).columns.tolist())       # drop COLUMNS with any missing

original shape: (6, 5)
(1, 5)
(6, 5)
(3, 5)
(4, 5)
['name']


### 5.3 Filling missing values

In [33]:
filled = messy.copy()

filled["age"] = filled["age"].fillna(filled["age"].median())       # median
filled["city"] = filled["city"].fillna("Unknown")                  # constant
filled["salary"] = filled["salary"].fillna(filled["salary"].mean().round(0))  # mean
filled["score"] = filled["score"].fillna(0)

print(filled)
print("missing left:", filled.isna().sum().sum())

  name   age       city   salary  score
0  Ana  23.0     Lahore  52000.0   88.0
1  Ben  31.5    Karachi  61000.0   92.0
2   Cy  28.0    Unknown  63800.0   75.0
3  Dia  35.0  Islamabad  75000.0    0.0
4  Eli  31.5    Karachi  49000.0    0.0
5  Fay  41.0    Unknown  82000.0   90.0
missing left: 0


### 5.4 Forward fill, backward fill, interpolation

In [34]:
ts = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan, 6.0],
               index=pd.date_range("2026-01-01", periods=6))

print(pd.DataFrame({
    "original": ts,
    "ffill": ts.ffill(),                     # carry last value forward
    "bfill": ts.bfill(),                     # pull next value backward
    "interpolate": ts.interpolate(),         # straight line between known points
}))

            original  ffill  bfill  interpolate
2026-01-01       1.0    1.0    1.0          1.0
2026-01-02       NaN    1.0    4.0          2.0
2026-01-03       NaN    1.0    4.0          3.0
2026-01-04       4.0    4.0    4.0          4.0
2026-01-05       NaN    4.0    6.0          5.0
2026-01-06       6.0    6.0    6.0          6.0


### 5.5 Fill with group-specific values

In [35]:
sales = pd.DataFrame({
    "region": ["N", "N", "S", "S", "S", "N"],
    "revenue": [100, np.nan, 200, np.nan, 220, 120],
})

# Replace missing revenue with the mean of its own region
sales["revenue"] = sales["revenue"].fillna(sales.groupby("region")["revenue"].transform("mean"))
print(sales)

  region  revenue
0      N    100.0
1      N    110.0
2      S    200.0
3      S    210.0
4      S    220.0
5      N    120.0


### 5.6 Replacing placeholder values and fixing types

In [36]:
raw = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "age": ["25", "N/A", "31", "-"],
    "country": ["PK", "", "US", "  "],
})

raw["age"] = raw["age"].replace({"N/A": np.nan, "-": np.nan})
raw["age"] = pd.to_numeric(raw["age"], errors="coerce")        # bad strings -> NaN

raw["country"] = raw["country"].str.strip().replace("", np.nan)
print(raw)
print(raw.dtypes)
print(raw.isna().sum())

   id   age country
0   1  25.0      PK
1   2   NaN     NaN
2   3  31.0      US
3   4   NaN     NaN
id           int64
age        float64
country        str
dtype: object
id         0
age        2
country    2
dtype: int64


### 5.7 Missing values in calculations

In [37]:
s = pd.Series([10, 20, np.nan, 40])

print(s.sum())                        # NaN is skipped by default
print(s.mean())
print(s.sum(skipna=False))            # nan if you don't skip
print(s.count(), "of", len(s), "values present")

print(np.nan == np.nan)               # NaN is never equal to itself, use isna()
print(pd.isna(np.nan))

70.0
23.333333333333332
nan
3 of 4 values present
False
True


---
## 6. Mini Project: Clean and Analyze a Messy Dataset

Combines inspecting, cleaning missing data, filtering, sorting, and grouping.

In [39]:
raw = pd.DataFrame({
    "customer": ["A", "B", "C", "D", "E", "F", "G", "H"],
    "region":   ["North", "South", None, "North", "East", "South", "East", None],
    "orders":   [5, 3, np.nan, 8, 2, np.nan, 7, 4],
    "spend":    ["250", "N/A", "120", "480", "90", "300", "-", "210"],
})

# 1) Inspect
print(raw.isna().sum(), "\n")

# 2) Clean
clean = raw.copy()
clean["spend"] = pd.to_numeric(clean["spend"], errors="coerce")
clean["region"] = clean["region"].fillna("Unknown")
clean["orders"] = clean["orders"].fillna(clean["orders"].median())
clean["spend"] = clean["spend"].fillna(clean["spend"].median())
print(clean, "\n")

# 3) Add a metric
clean["avg_order_value"] = (clean["spend"] / clean["orders"]).round(2)

# 4) Filter + sort
top = clean[clean["spend"] > 150].sort_values("avg_order_value", ascending=False)
print(top[["customer", "region", "spend", "avg_order_value"]], "\n")

# 5) Summarize
print(clean.groupby("region").agg(customers=("customer", "count"), total_spend=("spend", "sum"))
           .sort_values("total_spend", ascending=False))

customer    0
region      2
orders      2
spend       0
dtype: int64 

  customer   region  orders  spend
0        A    North     5.0  250.0
1        B    South     3.0  230.0
2        C  Unknown     4.5  120.0
3        D    North     8.0  480.0
4        E     East     2.0   90.0
5        F    South     4.5  300.0
6        G     East     7.0  230.0
7        H  Unknown     4.0  210.0 

  customer   region  spend  avg_order_value
1        B    South  230.0            76.67
5        F    South  300.0            66.67
3        D    North  480.0            60.00
7        H  Unknown  210.0            52.50
0        A    North  250.0            50.00
6        G     East  230.0            32.86 

         customers  total_spend
region                         
North            2        730.0
South            2        530.0
Unknown          2        330.0
East             2        320.0




### Gotchas to remember
- Use `&`, `|`, `~` with parentheses for conditions, not `and` / `or` / `not`.
- `loc` slices include the end label; `iloc` slices exclude the end position.
- Most methods **return a new object**; assign the result (`df = df.dropna()`) rather than relying on in-place changes.
- Use `.copy()` when you plan to modify a subset, to avoid chained-assignment surprises.
- `NaN != NaN`, so always test with `isna()`.

